# Etapa 2

Correr toda la etapa 2 para el análisis de resultados
- datos
- baselines
- búsqueda de hiperparámetros
- 5 semillas por modelo 
- evaluación 
- XAI.

GPU T4

## 1. Preparar el entorno

In [ ]:
import os

REPO = "natiorozco/HTR-SitelenPona"
RAMA = "etapa2"

if not os.path.exists("/content/HTR-SitelenPona"):
    try:
        # si el repo fuera privado
        from google.colab import userdata
        url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
    except Exception:
        url = f"https://github.com/{REPO}.git"
    !git clone -q -b {RAMA} {url} /content/HTR-SitelenPona

%cd /content/HTR-SitelenPona
!git log -1 --format='commit %h: %s (%an, %ad)' --date=short

In [ ]:

!pip install -q scikit-image

import platform
import keras, numpy, sklearn, skimage, tensorflow as tf, torch, torchvision
print("python", platform.python_version())
for m in (numpy, sklearn, skimage, torch, torchvision, tf, keras):
    print(m.__name__, m.__version__)
print("GPU torch:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
print("GPU tensorflow:", tf.config.list_physical_devices("GPU"))

In [ ]:
import glob
from pathlib import Path

faltan = [p for p in ["Etapa_2/tradeoff.py", "Etapa_2/xai/gradcam_torch.py", "Etapa_2/xai/gradcam_tf.py"]
          if not Path(p).exists()]
print("Faltan en el repo:", faltan if faltan else "nada")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RESPALDO = "/content/drive/MyDrive/HTR-SitelenPona-etapa2"

def respaldar():
    # -u copia solo lo nuevo o cambiado, para no volver a subir todos los modelos cada vez
    os.makedirs(RESPALDO, exist_ok=True)
    !cp -ru Etapa_2/results Etapa_2/data/processed "{RESPALDO}/"

def ya_corrio(log):
    """True si el log ya existe (corrida terminada antes de que Colab se desconectara)."""
    hecho = Path(f"Etapa_2/results/logs/{log}.json").exists()
    if hecho:
        print(f"ya estaba: {log}")
    return hecho

## 2. Datos

Se borran los resultados viejos del repo y, si Colab se desconectó, se recupera de Drive lo que ya había corrido: las celdas de abajo se saltan las corridas que ya tienen log.

El split no se regenera aquí: viene en el repo, generado con las versiones de `requirements.txt`. Con otra versión de scikit-learn `prepare.py` da otra partición (en Colab salió con test de 5 %), así que esta celda solo comprueba que el split del repo esté bien.

In [ ]:
!rm -rf Etapa_2/results
if os.path.exists(f"{RESPALDO}/results"):
    !cp -r "{RESPALDO}/results" Etapa_2/
    for sub in ("logs", "logs/busqueda", "preds", "modelos"):
        print(f"recuperado de Drive, {sub}: {len(glob.glob(f'Etapa_2/results/{sub}/*.*'))} archivos")

import csv

rows = list(csv.DictReader(open("Etapa_2/data/processed/splits.csv")))
palabras = {r["clase"] for r in rows}
for sp in ("train", "val", "test"):
    sub = [r for r in rows if r["split"] == sp]
    print(f"{sp:5s} {len(sub):5d} ({len(sub) / len(rows):.1%}), palabras: {len({r['clase'] for r in sub})}")
    assert {r["clase"] for r in sub} == palabras, f"faltan palabras en {sp}"
print(len(palabras), "palabras")

# lo recuperado de la primera corrida (con el split malo) no sirve: se borra para que se vuelva a correr
split_de = {r["id"]: r["split"] for r in rows}
for pred in glob.glob("Etapa_2/results/preds/*.csv"):
    if any(split_de.get(r["id"]) != r["split"] for r in csv.DictReader(open(pred))):
        corrida = Path(pred).stem
        print("split viejo, se descarta:", corrida)
        for f in [pred, f"Etapa_2/results/logs/{corrida}.json", f"Etapa_2/results/logs/{corrida.removesuffix('_seed0')}.json"]:
            Path(f).unlink(missing_ok=True)
respaldar()

## 3. Baselines

In [ ]:
if not glob.glob("Etapa_2/results/preds/dummy_*.csv"):
    !python Etapa_2/baselines/dummy.py
for f in ("pixels", "hog"):
    if not ya_corrio(f"{f}_lr"):
        !python Etapa_2/baselines/hog_lr.py --features {f}
respaldar()

## 4. Búsqueda de hiperparámetros (PyTorch)

Una semilla, solo validación. Cada corrida queda en `results/logs/busqueda/` y no escribe predicciones.
Después se elige la configuración con mejor macro-F1 de validación.

In [ ]:
import itertools

BUSQUEDA = {
    "cnn_sinaug":   {"base": "--model cnn --aug none",
                     "grilla": {"lr": [3e-4, 1e-3, 3e-3], "batch": [64, 128], "freeze-epochs": [0]}},
    "cnn_aug":      {"base": "--model cnn --aug trad",
                     "grilla": {"lr": [3e-4, 1e-3, 3e-3], "batch": [64, 128], "freeze-epochs": [0]}},
    "resnet18_aug": {"base": "--model resnet18 --aug trad",
                     "grilla": {"lr": [1e-4, 3e-4, 1e-3], "batch": [64], "freeze-epochs": [0, 3]}},
}

for nombre, b in BUSQUEDA.items():
    for lr, batch, fr in itertools.product(*b["grilla"].values()):
        # mismo nombre de log que arma train.py con --search (weight decay por defecto 1e-4)
        if ya_corrio(f"busqueda/{nombre}_seed0_lr{lr}_b{batch}_wd0.0001_f{fr}"):
            continue
        extra = f"--lr {lr} --batch {batch} --freeze-epochs {fr}"
        print(f"\n### {nombre}: {extra}")
        !python Etapa_2/torch_models/train.py {b["base"]} {extra} --seed 0 --search | tail -2
        respaldar()

In [ ]:
import glob
import json

def mejor_config(nombre):
    logs = [json.load(open(p)) for p in glob.glob(f"Etapa_2/results/logs/busqueda/{nombre}_seed0_*.json")]
    for l in sorted(logs, key=lambda l: -l["mejor_val_macro_f1"]):
        c = l["config"]
        print(f"  lr={c['lr']:<7} batch={c['batch']:<4} freeze={c['freeze_epochs']}  "
              f"val macro-F1={l['mejor_val_macro_f1']:.4f}  ({l['segundos_entrenamiento']:.0f} s)")
    c = max(logs, key=lambda l: l["mejor_val_macro_f1"])["config"]
    return f"--lr {c['lr']} --batch {c['batch']} --freeze-epochs {c['freeze_epochs']}"

ELEGIDA = {}
for nombre in BUSQUEDA:
    print(nombre)
    ELEGIDA[nombre] = mejor_config(nombre)
    print("  elegida:", ELEGIDA[nombre], "\n")

## 5. Corridas finales: 5 semillas

Primero PyTorch con la configuración elegida. HCR-Net usa la configuración oficial (épocas por fase y
calendario de lr del paper), así que no lleva búsqueda: se reproduce tal cual como baseline de literatura.

In [ ]:
SEMILLAS = [0, 1, 2, 3, 4]

for nombre, b in BUSQUEDA.items():
    for s in SEMILLAS:
        if ya_corrio(f"{nombre}_seed{s}"):
            continue
        print(f"\n### {nombre} semilla {s}")
        !python Etapa_2/torch_models/train.py {b["base"]} {ELEGIDA[nombre]} --seed {s} | tail -2
        respaldar()

In [ ]:
HCRNET = {"hcrnet_sinaug": "--aug none", "hcrnet_aug": "--aug trad", "hcrnet_aug_elastica": "--aug trad --elastica"}

for nombre, config in HCRNET.items():
    for s in SEMILLAS:
        if ya_corrio(f"{nombre}_seed{s}"):
            continue
        print(f"\n### {nombre} semilla {s}")
        !python Etapa_2/hcrnet/train_hcrnet.py {config} --seed {s} | tail -2
        respaldar()

## 6. Evaluación y trade-off

In [ ]:
!python Etapa_2/evaluate.py --split val
!python Etapa_2/evaluate.py --split test
if Path("Etapa_2/tradeoff.py").exists():
    !python Etapa_2/tradeoff.py
respaldar()

## 7. XAI (Grad-CAM)

In [ ]:
if Path("Etapa_2/xai/gradcam_torch.py").exists():
    !python Etapa_2/xai/gradcam_torch.py --model cnn --aug none --seed 0
    !python Etapa_2/xai/gradcam_torch.py --model cnn --aug trad --seed 0
    !python Etapa_2/xai/gradcam_torch.py --model resnet18 --aug trad --seed 0
if Path("Etapa_2/xai/gradcam_tf.py").exists():
    !python Etapa_2/xai/gradcam_tf.py --run hcrnet_aug --seed 0
respaldar()

## 8. Descargar

Zip con `results/` y `data/processed/`.

In [ ]:
!zip -qr /content/etapa2_resultados.zip Etapa_2/results Etapa_2/data/processed
from google.colab import files
files.download("/content/etapa2_resultados.zip")